In [1]:
%run 05_MNESIS_parameters.ipynb

datetag = '2026-04-21'
SNNtorch version 0.9.4
Spikes in one target 1024.0,  in a SM window 42.0
for a value opt.lif_beta=0.7, the time constant is 2.8 steps


# generative model for spiking targets

In [ ]:
class SpikingPattern:

    def __init__(self):
        self.desc = " A simple pattern generator"

    def init(self, opt):
        self.opt = opt
        target_generator = torch.Generator()
        target_generator.manual_seed(opt.seed)
        p_bias = opt.p_A * torch.ones((opt.N_pattern, opt.N_neuron, opt.N_time))
        self.target = torch.bernoulli(p_bias, generator=target_generator)
        self.target = self.target.float()
        self.target = self.target.to(opt.device)

    def __call__(self):
        return self.target
            


# generative model for spiking motifs

In [ ]:
# class StochasticSpikingPattern(SpikingPattern):

#     def __init__(self, epsilon=1e-12):
#         super().__init__()
# #         self.opt = opt
# #         self.init()

#     def init(self):
#         """
        
#         initializes "temporal mod", which governs the overall structure of the motif. 
#         The shape is the product  of two exponenential functions creating a "psp-like" 
#         shape. in set_SM the temporal mod will cause each SM to have the 
#         distribution shape of temporal mod, such that the probability distribution 
#         of spike occurences has a fast rise and a slow decay.

#         """
#         self.logit_p_A = torch.logit(torch.tensor(self.opt.p_A))

#         # temporal_mod = torch.zeros(self.opt.num_delays)
#         self.time = torch.linspace(0, self.opt.T_trial, self.opt.N_time)
#         self.SM_time = torch.linspace(0, self.opt.T_trial, self.opt.num_delays)
#         # temporal_mod = torch.exp(- self.SM_time / self.opt.tau_decay)
#         # temporal_mod *= 1 - torch.exp(- self.SM_time / self.opt.tau_rise)
#         # temporal_mod = temporal_mod.flip([0])
#         #self.temporal_mod = (torch.eye(self.opt.N_SMs).unsqueeze(2)) * (temporal_mod.unsqueeze(0).unsqueeze(0))
#         # self.temporal_mod = torch.ones((self.opt.N_pre, self.opt.N_SMs, 1)) * (temporal_mod.unsqueeze(0).unsqueeze(0))
        
#         """
#         torch.eye -> identity matrix 
#         torch.unsqueeze -> wrap in more brackets
#         torch.flip -> reverses the orders of the elements in the vector
        
#         this creates a 144 by 144 by 3 matrix, 
#         Then at each diagonal there is an identical spike waveform
#         such that convolving the SM with that matrix will result in a
#         spike waveform at each presynptic input.
        
#         """
#         # spike = torch.tensor([1, -.6, -.25, -.10, -.05]) # raw spike shape with depolariation (1) and hyperpolarization ([-.8, -.2])
#         spike = torch.tensor([1, -.8, -.2]) # raw spike shape with depolariation (1) and hyperpolarization ([-.8, -.2])
#         self.spike = (torch.eye(self.opt.N_SMs).unsqueeze(2)) * (spike.flip([0]).unsqueeze(0).unsqueeze(0))


#     def set_SM(self, seed=None, seed_offset=0):
#         """
#         Returns a N_SMs by N_pre by num_delays dimensional matrix representing the probability distribution of each elemenet
#         in the matrix to evoke a spike. Presumably later spikes will be drawn from this distribution via Bernoulli trials.
#         """
#         if seed is None: seed = self.opt.seed + seed_offset
#         torch.manual_seed(seed)
        
#         # 1/ define SMs as matrices to be used as kernels
#         SM = self.opt.E_SM * torch.randn(self.opt.N_pre, self.opt.N_SMs, self.opt.num_delays)
#         #threshold = torch.abs(SM).quantile(1-self.opt.p_SM)

#         # 2/ zero out everything below the threshold and on the border
#         # TODO : get analytically
#         from scipy.stats import norm
#         threshold = self.opt.E_SM * norm.ppf(1-self.opt.p_SM)
#         SM *= (SM > threshold)

#         SM[:, :, :(self.spike.shape[-1]//2)] = 0
#         SM[:, :, -(self.spike.shape[-1]//2):] = 0
#         # 3/ modulate in time
#         # SM *= self.temporal_mod

#         # 4/ convolve with a spike shape to induce some sort of refractory period
#         SM = torch.conv1d(SM, self.spike, padding=self.spike.shape[-1]//2)

#         return torch.swapaxes(SM, 0, 1)

#     def get_b(self, seed=None, seed_offset=1):
#         """
#         generates a matrix of events for each SM, for each trial, returns a matrix of N_trials, N_SMs, N_time, 
#         draw causes (SMs) as a matrix of sparse SM activations uniformly in postsynaptic space and time
        
#         a 1 indicates the start of a spiking motif

#         """
#         if seed is None: seed = self.opt.seed + seed_offset
#         torch.manual_seed(seed)
        
#         # we set it to zero everywhere
#         b_proba = torch.zeros(self.opt.N_trials, self.opt.N_SMs, self.opt.N_time)
#         # set to p_B except outside the borders
#         b_proba[:, :, :-(self.opt.num_delays)] = self.opt.p_B        
#         return torch.bernoulli(b_proba)

#     def plot_raster(self, raster, raster_post=None, SM=None, 
#                     i_trial=0, shift=0, xticks=6, yticks=16, spikelength=.9, 
#                     colors=None, figsize=None, subplotpars=subplotpars, 
#                     ylabel='address', linewidths=1.0):
#         """    
#         plots b, which is trials by M by T
#         """

#         N_neurons = raster.shape[1]
#         if colors is None: # blue if nothing assigned
#             colors = ['b'] * N_neurons
#         else: # give the colors or ...
#             if len(colors)==1: # ... paint everything the same color
#                 colors = colors[0] * N_neurons

#         if figsize is None: figsize = (self.opt.fig_width, self.opt.fig_width/self.opt.phi)

#         fig, ax = plt.subplots(1, 1, figsize=figsize, subplotpars=subplotpars)
#         if raster_post is None:
#             for i in range(0, N_neurons):
#                 ax.eventplot(np.where(raster[i_trial, i, :] == 1.)[0]+shift, 
#                     colors=colors[i], lineoffsets=1.*i+spikelength/2, 
#                     linelengths=spikelength, linewidths=linewidths)
#         else:
#             for i_SM in range(self.opt.N_SMs):
#                 b_ = torch.zeros_like(raster_post)
#                 b_[i_trial, i_SM, :] = raster_post[i_trial, i_SM, :]
#                 a_ = self.draw_a(b_, SM)
#                 for i in range(0, self.opt.N_pre):
#                     ax.eventplot(np.where(a_[i_trial, i, :] == 1.)[0]+shift, colors=colors[i_SM], lineoffsets=1.*i+spikelength/2, 
#                     linelengths=spikelength, linewidths=linewidths)
    
#         ax.set_ylabel(ylabel)
#         ax.set_xlabel('Time (a. u.)')
#         ax.set_xlim(0, self.opt.N_time)
#         ax.set_ylim(0, N_neurons)

#         ax.yaxis.set_minor_locator(MultipleLocator(N_neurons//8))
#         ax.set_yticks(np.linspace(0, N_neurons, yticks, endpoint=False)+.5)
#         ax.set_yticklabels(np.linspace(1, N_neurons, yticks, endpoint=True).astype(int))
#         for side in ['top', 'right']: ax.spines[side].set_visible(False)
        
#         ax.xaxis.set_minor_locator(MultipleLocator(self.opt.N_time//8))
#         ax.set_xticks(np.linspace(1, self.opt.N_time, xticks, endpoint=True))
#         ax.set_xticklabels(np.linspace(1, self.opt.N_time, xticks, endpoint=True).astype(int))

#         ax.grid(visible=True, axis='y', linestyle='-', lw=.5)
#         ax.grid(visible=True, axis='x', which='both', linestyle='-', lw=.1)

#         return fig, ax


#     def plot_b(self, b, i_trial=0, xticks=6, yticks=16, spikelength=.9, colors=None, figsize=None, subplotpars=subplotpars):
#         fig, ax = self.plot_raster(raster=b, i_trial=i_trial, xticks=xticks, yticks=yticks, spikelength=spikelength, colors=colors, figsize=figsize, subplotpars=subplotpars, ylabel='@ Motif')
#         return fig, ax

#     def model_a_logit(self, b, SM):
#         """

#         Generative model:
#         defines the evidence of firing for each presynaptic address over time
#         returns the logit_p_A rasters for each trial

#         input b -> postsynaptic raster plot
#             SM -> spiking motifs as spatio temporal kernels
#             p_A -> prior proba of firing for the presynaptic addresses
        
#         """

#         logit_p_A_fromB = torch.conv_transpose1d(b*1., SM, padding=self.opt.num_delays//2)
#         logit_p_A_fromB = torch.roll(logit_p_A_fromB, self.opt.num_delays//2, dims=-1)
#         return self.logit_p_A + logit_p_A_fromB

#     def model_a(self, b, SM):
#         """

#         Generative model:
#         defines the probability of firing for each presynaptic address over time from its evidence

#         calls model_a_logit(b,SM) to get the logit, and applies the sigmoid over the logit_p_A rasters to turn the 
#         rasters into probabilities between 1 and 0        
#         """
#         logit_A = self.model_a_logit(b, SM)
#         return torch.sigmoid(logit_A)

#     def draw_a(self, b, SM, seed=None, seed_offset=2):
#         """
#         calls model_a(b,SM) and generates A_proba which is the probability distribution of neuron spiking events,
#         then this gets passed through a bernoulli trial, deciding where neurons spike given the probability distribution
#         """
#         # draws binary events from the probability of firing
#         if seed is None: seed = self.opt.seed + seed_offset
#         torch.manual_seed(seed)
#         # generate the corresponding raster plot
#         A_proba = self.model_a(b, SM)
#         return torch.bernoulli(A_proba)

#     def generative_model(self, seed=None, seed_offset=3):
#         if seed is None: seed = self.opt.seed + seed_offset
#         torch.manual_seed(seed)
#         SM, b = self.set_SM(seed=seed), self.get_b(seed=seed+1)
#         a = self.draw_a(b, SM, seed=seed+2)
#         return a, b, SM

#     def plot_a(self, a, b=None, SM=None, i_trial=0, shift=0, xticks=6, yticks=16, spikelength=.9, colors=None, figsize=None, subplotpars=subplotpars):
#         fig, ax = self.plot_raster(raster=a, raster_post=b, SM=SM, i_trial=i_trial, shift=shift,
#                                    xticks=xticks, yticks=yticks, spikelength=spikelength, colors=colors, 
#                                    figsize=figsize, subplotpars=subplotpars, ylabel='@ Neuron')
#         return fig, ax

#     def do_thresholding(self, b, p_B, max_quant=10000000):
#         """

#         Threshold model:

#         essentially this function transforms the convoluted matrix which is unnormalized and unbounded into
#         a matrix of quantiles, and then binarizes the matrix by thresholding. The function returns the 
#         binary b matrix...
        
#         """
#         if len(b.ravel()) > max_quant: # the quantile function does not like having to many samples
#             ind_quant = torch.randperm(len(b.ravel()))[:max_quant]
#             b_threshold = torch.quantile(b.ravel()[ind_quant], 1-p_B)
#         else:
#             b_threshold = torch.quantile(b, 1-p_B)

#         b_hat_bin = (b > b_threshold) * 1.
#         return b_hat_bin

#     def test_model(self, SM, SM_true=None, seed=None, seed_offset=4, method='HD-SNN'):
#         """
#         this function calls inference_with_SMs(a,b,SM). This function calls get_b and draw_a over again, so 
#         if this method is called it will reset the raster plot given SM... ( i think?)... hmmmmm
#         Or maybe SM are the priors?   
        
#         """
#         if seed is None: seed = self.opt.seed + seed_offset
#         torch.manual_seed(seed)
#         if SM_true is None: SM_true = SM
#         # define SMs
#         # draw causes (SMs)
#         b = self.get_b(seed=seed)
#         # generate the corresponding raster plot
#         a = self.draw_a(b, SM_true, seed=seed+1)
#         # infer 
#         _, b_hat_bin = self.inference_with_SMs(a, b, SM, method=method)
#         # count
#         accuracy = torch.mean((b_hat_bin == b)*1.)
#         TP = torch.mean(b_hat_bin[b==1]*1.)
#         TN = 1-torch.mean(b_hat_bin[b==0]*1.)
#         return accuracy, TP, TN
        
#     def plot_SM(self, SM, cmap='seismic', colors=None, aspect=None, figsize=None, subplotpars=subplotpars, N_SM_show: int=0):

#         if N_SM_show == 0: N_SM_show = self.opt.N_SM_show
#         if SM.dtype == torch.bool:
#             SM_max = 1
#             SM_min = 0
#             cmap = 'binary'
#         else:
#             # SM = SM.numpy()
#             SM_max = np.abs(SM).max()#.item()
#             SM_min = -SM_max

#         if figsize is None: figsize = (self.opt.fig_width, self.opt.fig_width/self.opt.phi)

#         fig, axs = plt.subplots(1, N_SM_show, figsize=figsize, subplotpars=subplotpars)
#         for i_SM in range(N_SM_show):
#             ax = axs[i_SM]
#             ax.set_axisbelow(True)

#             ax.pcolormesh(SM[i_SM, :, :], cmap=cmap, vmin=SM_min, vmax=SM_max)
#             #ax.imshow(SM[:, i_SM, :], cmap=cmap, vmin=SM_min, vmax=SM_max, interpolation='none')
            
#             ax.set_xlim(0, SM.shape[2])

#             ax.set_xlabel('Delay')
#             # ax.set_title(f'Motif {i_SM+1}', color='k' if colors is None else colors[i_SM])
#             # ax.text(-25, -.2*self.opt.N_pre, f'#{i_SM+1}', color='k' if colors is None else colors[i_SM])
#             t = ax.text(.40*self.opt.num_delays, .95*self.opt.N_pre, f'SM #{i_SM+1}', color='k' if colors is None else colors[i_SM])
#             t.set_bbox(dict(facecolor='white', edgecolor='white'))
#             if not aspect is None: ax.set_aspect(aspect)

#             ax.set_ylim(0, self.opt.N_pre)
#             # ax.yaxis.set_minor_locator(MultipleLocator(self.opt.N_pre//8))
#             ax.set_yticks(np.arange(0, self.opt.N_pre, self.opt.N_pre//8)+.5)
#             if i_SM>0: 
#                 ax.set_yticklabels([])
#             else:
#                 ax.set_yticklabels(np.arange(0, self.opt.N_pre, self.opt.N_pre//8)+1)

#             for side in ['top', 'right']: ax.spines[side].set_visible(False)
#             #ax.set_xticks([0, self.opt.num_delays//2, self.opt.num_delays-1])
#             ax.set_xticks([1, self.opt.num_delays//3, (self.opt.num_delays*2)//3, self.opt.num_delays-1])
#             # ax.set_xticklabels([0, (self.opt.num_delays//2), (self.opt.num_delays)])
#             ax.xaxis.set_minor_locator(AutoMinorLocator(self.opt.num_delays//4))
#             #ax.xaxis.set_minor_locator(AutoMinorLocator(4))
#             #ax.set_xticklabels([-(self.opt.num_delays//2), 0, self.opt.num_delays//2])
#             # ax.grid(True, axis='y', linestyle='-', lw=1)
#             # ax.grid(True, axis='x', which='both', linestyle='-', lw=.1)

#         axs[0].set_ylabel('@ Neuron')
#         return fig, axs


In [4]:
# opt = Params()
# env = StochasticSpikingPattern(opt)
# a, b, SM_true = env.generative_model()
# env.plot_SM(SM_true)
# SM_true.shape

Draw one instance of single SMs:

Draw the occurrences of SMs:

In [5]:
i_trial = 0
# for i_neuron in range(opt.N_SMs):
#     print(i_neuron, torch.nonzero(b[i_trial, i_neuron, :]).numpy())

In [6]:
# time_mark = 283
# fig, ax = env.plot_b(b, i_trial=i_trial)
# ax.vlines([time_mark], 0, opt.N_SMs, 'r', linestyles='-.', alpha=.5)

Resulting raster plot:

In [7]:
# fig, ax = env.plot_a(a, i_trial=i_trial)
# ax.vlines([time_mark], 0, opt.N_pre, 'r', linestyles='-.', alpha=.5)
# ax.vlines([time_mark + opt.num_delays], 0, opt.N_pre, 'g', linestyles='-.', alpha=.5)
# a.shape

TODO: circular padding?

In [8]:
# a.mean(), b.mean(), b.mean(axis=(0, -1))[:20]

In [9]:
# b.shape

In [10]:
# b[i_trial, 0, :]

In [11]:
# for i_trial in range(opt.N_trials):
#     print(i_trial, torch.nonzero(b[i_trial, i_neuron, :]).numpy())